<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/ML/W12/Q1/ASSIGNMNT_W12_Q_1_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# TASK 1: SPECTRAL DATA TRANSFORMATION AND CLUSTER ANALYSIS
# IIT Jodhpur BSc - Introduction to Databases / Data Analysis
#
# Generates:
# 1. Spectral_Clustering_Calculation_Sheet.pdf
# 2. Similarity_Matrix_W.png
# 3. Degree_Matrix_D.png
# 4. Laplacian_Matrix_L.png
# 5. Eigenvalues_Eigenvectors.png
# 6. Spectral_Embedding.png
# 7. Eigenvalues_Eigenvectors.csv
# 8. Spectral_Embedding_Coordinates.csv
# 9. Task_1_Spectral_Clustering.zip
# ============================================================

!pip -q install numpy pandas matplotlib reportlab

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, Image, PageBreak, Preformatted
)
from reportlab.lib.units import inch
from google.colab import files

# ------------------------------------------------------------
# 1. CREATE OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = "/content/Task_1_Spectral_Clustering"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. DEFINE THE FOUR-POINT DATASET
# ------------------------------------------------------------

points = np.array([
    [0.0, 0.0],   # x1
    [0.5, 0.0],   # x2
    [5.0, 5.0],   # x3
    [5.5, 5.0]    # x4
])

point_names = ["x1", "x2", "x3", "x4"]
actual_groups = ["Group 1", "Group 1", "Group 2", "Group 2"]

sigma = 1.0
radius = 1.0

n = len(points)

# ------------------------------------------------------------
# 3. CALCULATE SQUARED EUCLIDEAN DISTANCES
# ------------------------------------------------------------

squared_distances = np.zeros((n, n))
distances = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        squared_distances[i, j] = np.sum((points[i] - points[j]) ** 2)
        distances[i, j] = np.sqrt(squared_distances[i, j])

# ------------------------------------------------------------
# 4. CREATE SIMILARITY MATRIX W
#
# Gaussian similarity:
# W(i,j) = exp(-||xi-xj||^2 / (2*sigma^2))
#
# Radius graph:
# Similarity is retained only when distance <= radius.
# This creates two disconnected components.
# ------------------------------------------------------------

W = np.zeros((n, n))

for i in range(n):
    for j in range(n):

        if i == j:
            W[i, j] = 0.0

        elif distances[i, j] <= radius:
            W[i, j] = np.exp(
                -squared_distances[i, j] / (2 * sigma**2)
            )

        else:
            W[i, j] = 0.0

# ------------------------------------------------------------
# 5. CALCULATE DEGREE MATRIX D
# ------------------------------------------------------------

degrees = np.sum(W, axis=1)
D = np.diag(degrees)

# ------------------------------------------------------------
# 6. CALCULATE GRAPH LAPLACIAN
#
# L = D - W
# ------------------------------------------------------------

L = D - W

# ------------------------------------------------------------
# 7. CALCULATE EIGENVALUES AND EIGENVECTORS
# ------------------------------------------------------------

calculated_eigenvalues, calculated_eigenvectors = np.linalg.eigh(L)

# Sort eigenvalues and corresponding eigenvectors
order = np.argsort(calculated_eigenvalues)

calculated_eigenvalues = calculated_eigenvalues[order]
calculated_eigenvectors = calculated_eigenvectors[:, order]

# Because there are two disconnected components, the zero
# eigenvalue has multiplicity 2.
#
# We use a clear, deterministic orthonormal eigenbasis.
# This avoids arbitrary rotations within the zero-eigenvalue
# subspace returned by numerical eigensolvers.

w12 = W[0, 1]
w34 = W[2, 3]

eigenvalues = np.array([
    0.0,
    0.0,
    2 * w12,
    2 * w34
])

eigenvectors = np.array([
    [1/np.sqrt(2),  0,              1/np.sqrt(2),  0],
    [1/np.sqrt(2),  0,             -1/np.sqrt(2),  0],
    [0,              1/np.sqrt(2),  0,              1/np.sqrt(2)],
    [0,              1/np.sqrt(2),  0,             -1/np.sqrt(2)]
])

# Each column is one eigenvector.
# Verify the eigenvalue equation L*v = lambda*v.

for i in range(4):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]

    assert np.allclose(L @ v, lam * v, atol=1e-8), \
        f"Eigenvector verification failed for v{i+1}"

# ------------------------------------------------------------
# 8. CREATE TWO-DIMENSIONAL SPECTRAL EMBEDDING
#
# Use the first two eigenvectors corresponding to
# the two smallest eigenvalues.
# ------------------------------------------------------------

embedding = eigenvectors[:, :2]

embedding_df = pd.DataFrame(
    embedding,
    columns=["Spectral_Dimension_1", "Spectral_Dimension_2"]
)

embedding_df.insert(0, "Point", point_names)
embedding_df["Expected_Group"] = actual_groups

# ------------------------------------------------------------
# 9. CREATE DATA TABLES
# ------------------------------------------------------------

points_df = pd.DataFrame(
    points,
    columns=["X", "Y"],
    index=point_names
)

distance_df = pd.DataFrame(
    distances,
    index=point_names,
    columns=point_names
)

squared_distance_df = pd.DataFrame(
    squared_distances,
    index=point_names,
    columns=point_names
)

W_df = pd.DataFrame(
    W,
    index=point_names,
    columns=point_names
)

D_df = pd.DataFrame(
    D,
    index=point_names,
    columns=point_names
)

L_df = pd.DataFrame(
    L,
    index=point_names,
    columns=point_names
)

# ------------------------------------------------------------
# 10. SAVE CSV FILES
# ------------------------------------------------------------

points_df.to_csv(os.path.join(OUTPUT_DIR, "Input_Dataset.csv"))

distance_df.to_csv(os.path.join(OUTPUT_DIR, "Euclidean_Distance_Matrix.csv"))

squared_distance_df.to_csv(
    os.path.join(OUTPUT_DIR, "Squared_Distance_Matrix.csv")
)

W_df.to_csv(os.path.join(OUTPUT_DIR, "Similarity_Matrix_W.csv"))

D_df.to_csv(os.path.join(OUTPUT_DIR, "Degree_Matrix_D.csv"))

L_df.to_csv(os.path.join(OUTPUT_DIR, "Laplacian_Matrix_L.csv"))

eigen_df = pd.DataFrame({
    "Eigenvalue": eigenvalues,
    "Eigenvector": [
        str(eigenvectors[:, i].round(6).tolist())
        for i in range(4)
    ]
})

eigen_df.to_csv(
    os.path.join(OUTPUT_DIR, "Eigenvalues_Eigenvectors.csv"),
    index=False
)

embedding_df.to_csv(
    os.path.join(OUTPUT_DIR, "Spectral_Embedding_Coordinates.csv"),
    index=False
)

# ------------------------------------------------------------
# 11. DISPLAY CALCULATIONS
# ------------------------------------------------------------

print("=" * 65)
print("TASK 1: SPECTRAL DATA TRANSFORMATION AND CLUSTER ANALYSIS")
print("=" * 65)

print("\nINPUT DATASET")
print(points_df)

print("\nSQUARED EUCLIDEAN DISTANCE MATRIX")
print(squared_distance_df.round(6))

print("\nSIMILARITY MATRIX W")
print(W_df.round(6))

print("\nDEGREE MATRIX D")
print(D_df.round(6))

print("\nLAPLACIAN MATRIX L = D - W")
print(L_df.round(6))

print("\nEIGENVALUES")
for i, value in enumerate(eigenvalues):
    print(f"Lambda {i+1} = {value:.6f}")

print("\nEIGENVECTORS (COLUMNS)")
for i in range(4):
    print(f"v{i+1} =", eigenvectors[:, i].round(6))

print("\nSPECTRAL EMBEDDING")
print(embedding_df.round(6))

# ------------------------------------------------------------
# 12. GENERATE MATRIX VISUALIZATION FUNCTION
# ------------------------------------------------------------

def save_matrix_plot(matrix, title, filename, labels):
    fig, ax = plt.subplots(figsize=(6, 5))

    im = ax.imshow(matrix, cmap="Blues")

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))

    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(
                j, i,
                f"{matrix[i, j]:.3f}",
                ha="center",
                va="center",
                fontsize=10
            )

    ax.set_title(title, fontsize=13)
    fig.colorbar(im, ax=ax)

    plt.tight_layout()

    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

    return path

# ------------------------------------------------------------
# 13. PLOT SIMILARITY, DEGREE AND LAPLACIAN MATRICES
# ------------------------------------------------------------

save_matrix_plot(
    W,
    "Similarity Matrix W",
    "Similarity_Matrix_W.png",
    point_names
)

save_matrix_plot(
    D,
    "Degree Matrix D",
    "Degree_Matrix_D.png",
    point_names
)

save_matrix_plot(
    L,
    "Graph Laplacian L = D - W",
    "Laplacian_Matrix_L.png",
    point_names
)

# ------------------------------------------------------------
# 14. PLOT ORIGINAL DATA
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5))

for i, point in enumerate(points):
    ax.scatter(
        point[0],
        point[1],
        s=120,
        label=actual_groups[i]
        if actual_groups[i] not in [
            actual_groups[j] for j in range(i)
        ] else ""
    )

    ax.annotate(
        point_names[i],
        (point[0], point[1]),
        xytext=(7, 7),
        textcoords="offset points"
    )

ax.set_title("Original Four-Point Dataset")
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.grid(True)
ax.legend()

plt.tight_layout()

original_plot = os.path.join(OUTPUT_DIR, "Original_Dataset.png")
plt.savefig(original_plot, dpi=200, bbox_inches="tight")
plt.show()
plt.close()

# ------------------------------------------------------------
# 15. PLOT EIGENVALUES AND EIGENVECTORS
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(
    ["λ1", "λ2", "λ3", "λ4"],
    eigenvalues
)

axes[0].set_title("Four Eigenvalues")
axes[0].set_ylabel("Eigenvalue")
axes[0].grid(axis="y")

im = axes[1].imshow(eigenvectors, cmap="coolwarm", aspect="auto")

axes[1].set_xticks(range(4))
axes[1].set_xticklabels(["v1", "v2", "v3", "v4"])
axes[1].set_yticks(range(4))
axes[1].set_yticklabels(point_names)

axes[1].set_title("Eigenvector Matrix")

for i in range(4):
    for j in range(4):
        axes[1].text(
            j, i,
            f"{eigenvectors[i, j]:.3f}",
            ha="center",
            va="center"
        )

fig.colorbar(im, ax=axes[1])

plt.tight_layout()

eigen_plot = os.path.join(
    OUTPUT_DIR,
    "Eigenvalues_Eigenvectors.png"
)

plt.savefig(eigen_plot, dpi=200, bbox_inches="tight")
plt.show()
plt.close()

# ------------------------------------------------------------
# 16. PLOT SPECTRAL EMBEDDING
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 6))

for i in range(4):
    ax.scatter(
        embedding[i, 0],
        embedding[i, 1],
        s=160,
        label=actual_groups[i]
        if actual_groups[i] not in [
            actual_groups[j] for j in range(i)
        ] else ""
    )

    ax.annotate(
        point_names[i],
        (embedding[i, 0], embedding[i, 1]),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=11
    )

ax.set_title("Two-Dimensional Spectral Representation")
ax.set_xlabel("First Eigenvector (v1)")
ax.set_ylabel("Second Eigenvector (v2)")
ax.grid(True)
ax.legend()

plt.tight_layout()

embedding_plot = os.path.join(
    OUTPUT_DIR,
    "Spectral_Embedding.png"
)

plt.savefig(embedding_plot, dpi=200, bbox_inches="tight")
plt.show()
plt.close()

# ------------------------------------------------------------
# 17. CREATE PDF CALCULATION SHEET
# ------------------------------------------------------------

pdf_path = os.path.join(
    OUTPUT_DIR,
    "Spectral_Clustering_Calculation_Sheet.pdf"
)

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=40,
    leftMargin=40,
    topMargin=40,
    bottomMargin=40
)

styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="MainTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=17,
    leading=22,
    textColor=colors.darkblue
))

styles.add(ParagraphStyle(
    name="SectionHeading",
    parent=styles["Heading2"],
    textColor=colors.darkblue,
    spaceBefore=12,
    spaceAfter=7
))

styles.add(ParagraphStyle(
    name="SmallText",
    parent=styles["BodyText"],
    fontSize=9,
    leading=13
))

story = []

story.append(Paragraph(
    "TASK 1: SPECTRAL DATA TRANSFORMATION AND CLUSTER ANALYSIS",
    styles["MainTitle"]
))

story.append(Spacer(1, 10))

story.append(Paragraph(
    "Objective: Transform a four-point dataset into a graph Laplacian, "
    "calculate its eigenvalues and eigenvectors, and represent the data "
    "in a lower-dimensional spectral space.",
    styles["BodyText"]
))

story.append(Paragraph("1. Input Dataset", styles["SectionHeading"]))

data = [["Point", "X", "Y", "Expected Group"]]

for i in range(4):
    data.append([
        point_names[i],
        str(points[i, 0]),
        str(points[i, 1]),
        actual_groups[i]
    ])

table = Table(data, hAlign="LEFT")

table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightblue),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("PADDING", (0, 0), (-1, -1), 7),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold")
]))

story.append(table)

story.append(Paragraph("2. Squared Euclidean Distance", styles["SectionHeading"]))

story.append(Paragraph(
    "The squared Euclidean distance between two points is:",
    styles["BodyText"]
))

story.append(Preformatted(
    "d²(xi, xj) = (xi1 - xj1)² + (xi2 - xj2)²",
    styles["Code"]
))

story.append(Paragraph(
    "For example, between x1 = (0, 0) and x2 = (0.5, 0):",
    styles["BodyText"]
))

story.append(Preformatted(
    "d²(x1, x2) = (0 - 0.5)² + (0 - 0)² = 0.25",
    styles["Code"]
))

story.append(Paragraph(
    "The squared distance matrix is:",
    styles["BodyText"]
))

story.append(Preformatted(
    np.array2string(squared_distances, precision=4),
    styles["Code"]
))

story.append(Paragraph("3. Similarity Matrix W", styles["SectionHeading"]))

story.append(Paragraph(
    "Gaussian similarity is calculated using:",
    styles["BodyText"]
))

story.append(Preformatted(
    "W(i,j) = exp(-d²(xi,xj) / (2*sigma²))",
    styles["Code"]
))

story.append(Paragraph(
    f"Sigma = {sigma}. A radius-based graph is used with radius = {radius}. "
    "An edge is retained only when the Euclidean distance is at most "
    "the specified radius. Diagonal values are zero.",
    styles["BodyText"]
))

story.append(Paragraph(
    f"For x1 and x2: W(1,2) = exp(-0.25/2) = {w12:.6f}.",
    styles["BodyText"]
))

story.append(Paragraph(
    f"For x3 and x4: W(3,4) = exp(-0.25/2) = {w34:.6f}.",
    styles["BodyText"]
))

story.append(Preformatted(
    "Similarity Matrix W:\n" +
    np.array2string(W, precision=6),
    styles["Code"]
))

story.append(Paragraph("4. Degree Matrix D", styles["SectionHeading"]))

story.append(Paragraph(
    "The degree of each vertex is the sum of the corresponding row "
    "in W. The degree matrix is diagonal.",
    styles["BodyText"]
))

story.append(Preformatted(
    "D = diag(sum of each row of W)\n\n" +
    np.array2string(D, precision=6),
    styles["Code"]
))

story.append(Paragraph("5. Graph Laplacian L = D - W", styles["SectionHeading"]))

story.append(Paragraph(
    "Subtracting the similarity matrix from the degree matrix gives "
    "the graph Laplacian:",
    styles["BodyText"]
))

story.append(Preformatted(
    "L = D - W\n\n" + np.array2string(L, precision=6),
    styles["Code"]
))

story.append(Paragraph(
    "The resulting Laplacian is block diagonal because there are "
    "no edges connecting the two pairs.",
    styles["BodyText"]
))

story.append(Preformatted(
    "L = [[ w12, -w12,    0,     0 ],\n"
    "     [-w12,  w12,    0,     0 ],\n"
    "     [   0,     0,  w34, -w34 ],\n"
    "     [   0,     0, -w34,  w34 ]]",
    styles["Code"]
))

story.append(Paragraph("6. Eigenvalues and Eigenvectors", styles["SectionHeading"]))

story.append(Paragraph(
    "The eigenvalue equation is L v = lambda v. "
    "The four eigenvalues and their corresponding normalized eigenvectors are:",
    styles["BodyText"]
))

eigen_table = [["Eigenvalue", "Corresponding Eigenvector"]]

for i in range(4):
    eigen_table.append([
        f"λ{i+1} = {eigenvalues[i]:.6f}",
        str(eigenvectors[:, i].round(6).tolist())
    ])

table = Table(eigen_table, colWidths=[130, 330])

table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightblue),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("PADDING", (0, 0), (-1, -1), 7),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, -1), 8)
]))

story.append(table)

story.append(Spacer(1, 10))

story.append(Paragraph(
    "There are two zero eigenvalues because the graph has two "
    "disconnected connected components. The first two eigenvectors "
    "form a valid two-dimensional spectral representation.",
    styles["BodyText"]
))

story.append(Paragraph("7. Lower-Dimensional Representation", styles["SectionHeading"]))

story.append(Paragraph(
    "The first two eigenvectors are used as the spectral coordinates. "
    "Each original point is represented by its corresponding row "
    "in the eigenvector matrix.",
    styles["BodyText"]
))

embed_table = [[
    "Point",
    "Dimension 1",
    "Dimension 2",
    "Group"
]]

for i in range(4):
    embed_table.append([
        point_names[i],
        f"{embedding[i, 0]:.6f}",
        f"{embedding[i, 1]:.6f}",
        actual_groups[i]
    ])

table = Table(embed_table)

table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightblue),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("PADDING", (0, 0), (-1, -1), 7),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold")
]))

story.append(table)

story.append(Spacer(1, 12))

story.append(Image(embedding_plot, width=5.5*inch, height=4.5*inch))

story.append(Paragraph("8. Conclusion", styles["SectionHeading"]))

story.append(Paragraph(
    "The graph Laplacian produces two zero eigenvalues, indicating "
    "two disconnected components. The first two eigenvectors represent "
    "the points in a two-dimensional spectral space. Points x1 and x2 "
    "share one spectral coordinate, while x3 and x4 share another. "
    "Therefore, the spectral representation separates the dataset "
    "into the two expected groups: (x1, x2) and (x3, x4).",
    styles["BodyText"]
))

story.append(Spacer(1, 10))

story.append(Paragraph(
    "Note: The four-point coordinates and radius threshold used here "
    "are illustrative because no original coordinates were supplied "
    "in the assignment description. Replace them with the official "
    "dataset if one was provided separately.",
    styles["SmallText"]
))

doc.build(story)

# ------------------------------------------------------------
# 18. CREATE ZIP FILE
# ------------------------------------------------------------

zip_path = "/content/Task_1_Spectral_Clustering.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    for filename in os.listdir(OUTPUT_DIR):

        full_path = os.path.join(OUTPUT_DIR, filename)

        if os.path.isfile(full_path):
            zipf.write(full_path, arcname=filename)

# ------------------------------------------------------------
# 19. FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SUCCESS: ALL DELIVERABLES GENERATED")
print("=" * 65)

print("\nEigenvalues:")
print(eigenvalues.round(6))

print("\nFinal spectral coordinates:")
print(embedding_df.round(6).to_string(index=False))

print("\nGenerated files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)

print("\nZIP FILE:")
print(zip_path)

print("\nDownloading ZIP file to your computer...")

files.download(zip_path)